# Phase 2 attention-intervention inspector

This notebook is the interactive architecture and intervention-mechanics companion to the Phase 2 runner. It does not change an architecture, write a checkpoint, or submit a job. Its optional GPT-OSS cell runs exactly one saved sample through the existing paired evaluator and displays both conditions; it is not a benchmark sweep.

> **Mechanics, not results.** The default tiny random CPU model proves selection and exact restoration. It does not demonstrate pretrained tool-selection behavior, model quality, or an experimental result.

Configuration-only views never construct or download a model. Any checkpoint load is explicit and uses a directory you supply on the machine or cluster where this notebook is running.

Run the setup cell, settings cell, imports, and inspector in order. `RUN_REAL_INTERVENTION_PROBE` is a mechanics-only safety check: it changes the selected tensors in memory, verifies the change, then verifies restoration without generating an answer. `RUN_PAIRED_SAMPLE_COMPARISON` is a one-query experiment: it runs an unchanged control and an enabled intervention through the GPT-OSS/Harmony evaluator and compares their raw outputs and outcomes.

In [ ]:
# Load local-only paths from .env, then make repository imports work.
# .env is ignored by Git. Copy .env.example to .env and set values for this machine.
import os
import sys
from pathlib import Path


def load_local_env(path):
    if not path.is_file():
        return False
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        name, value = line.split('=', 1)
        if name.isidentifier():
            os.environ.setdefault(name, value.strip())
    return True


env_file = next((folder / '.env' for folder in (Path.cwd(), *Path.cwd().parents) if (folder / '.env').is_file()), None)
if env_file is not None:
    load_local_env(env_file)

REPO_ROOT = os.environ.get('LAYERMCP_REPO_ROOT')
if REPO_ROOT:
    repository_root = Path(REPO_ROOT).expanduser().resolve()
else:
    repository_root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'research' / 'phase2' / 'intervention.py').is_file()), None)
if repository_root is None:
    raise RuntimeError('Repository root was not found. Open the notebook from its repository folder, or set LAYERMCP_REPO_ROOT in its ignored .env file.')
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))
CHECKPOINT_ENVIRONMENT = {
    'llama31': 'LAYERMCP_LLAMA31_8B_CHECKPOINT',
    'gpt_oss': 'LAYERMCP_GPT_OSS_CHECKPOINT',
    'qwen36': 'LAYERMCP_QWEN36_CHECKPOINT',
    'gemma4': 'LAYERMCP_GEMMA4_CHECKPOINT',
    'phi4': 'LAYERMCP_PHI4_CHECKPOINT',
}
print('repository root:', repository_root)
print('local .env:', env_file if env_file else 'not found')


In [ ]:
# Edit this cell before running a model-dependent cell. The sections below control different outputs.

# ── 1. Model and checkpoint loading ─────────────────────────────────────────
# Leave exactly one MODEL_CHOICE line uncommented.
MODEL_CHOICE = 'tiny_cpu'             # Small random CPU model; mechanics only.
# MODEL_CHOICE = 'gpt_oss_config_only'  # Real local config; never loads weights.
# MODEL_CHOICE = 'gpt_oss_checkpoint'   # Real local GPT-OSS weights; requires GPU.
# Choices: tiny_cpu, loaded_model, llama31_config_only, gpt_oss_config_only,
# qwen36_config_only, gemma4_config_only, phi4_config_only, or <family>_checkpoint.

# For <family>_checkpoint modes, CHECKPOINT_DIR comes from the mapped variable in ignored .env.
# Assign CHECKPOINT_DIR explicitly here only to temporarily override .env.
# No personal path is embedded here, and the notebook never downloads weights.
checkpoint_family = (
    MODEL_CHOICE.removesuffix('_checkpoint') if MODEL_CHOICE.endswith('_checkpoint')
    else MODEL_CHOICE.removesuffix('_config_only') if MODEL_CHOICE.endswith('_config_only')
    else None
)
CHECKPOINT_DIR = os.environ.get(CHECKPOINT_ENVIRONMENT[checkpoint_family]) if checkpoint_family else None

LOAD_CHECKPOINT = False  # Set True only for a <family>_checkpoint choice on a GPU.
LOADED_MODEL = None      # For loaded_model, assign a torch.nn.Module in an earlier cell.

# ── 2. Architecture inspection and target selection ─────────────────────────
# These settings drive the parameter table, acceptance check, mechanics probe, and one-sample pair.
LAYER_INDEX = 0
ATTENTION_TARGET = 'attention_all'
# ATTENTION_TARGET = 'qkv'  # GPT-OSS: fused query/key/value projection.
# ATTENTION_TARGET = 'out'
# Choose only a target printed as available: attention_all, q_proj, k_proj, v_proj,
# o_proj, qkv, or out. Available targets genuinely vary by family and layer.
RUN_ZERO_STRENGTH_ACCEPTANCE_PROBE = True  # Safety/type check without changing values.

# ── 3. Mechanics-only safety probe ──────────────────────────────────────────
# This never generates a benchmark answer. It uses LAYER_INDEX, ATTENTION_TARGET, and settings below.
RUN_REAL_INTERVENTION_PROBE = False
INTERVENTION_METHOD = 'noise'   # or 'replace'
INTERVENTION_STRENGTH = 0.01
INTERVENTION_SEED = 1234

# ── 4. One saved-sample paired experiment (GPT-OSS reasoning-low only) ──────
# This uses the model/inspection settings above. It runs one unchanged control and one intervention.
RUN_PAIRED_SAMPLE_COMPARISON = False
PAIRED_SAMPLE_ID = 'math_v1_calculator_easy_001'
PAIRED_SOURCE_RUN_DIR = os.environ.get('LAYERMCP_GPT_OSS_SOURCE_RUN')  # Shared by one-sample and sweep modes.

# ── 5. Bounded paired sweep (GPT-OSS reasoning-low only) ────────────────────
# This ignores INTERVENTION_* and LAYER_INDEX/ATTENTION_TARGET above; it uses only SWEEP_* settings.
# Use the literal 'all' for SWEEP_LAYERS and/or SWEEP_TARGETS only after reviewing the planned count.
RUN_PAIRED_SWEEP = False
SWEEP_SAMPLE_IDS = ('math_v1_calculator_easy_001',)
SWEEP_LAYERS = (0,)              # Tuple of layer IDs, or 'all'.
SWEEP_TARGETS = ('qkv',)         # Tuple of targets, or 'all' safe targets per layer.
SWEEP_METHODS = ('noise',)       # Any subset of ('noise', 'replace').
SWEEP_STRENGTHS = (0.001, 0.003, 0.01)
SWEEP_SEEDS = (1234,)
SWEEP_MAX_INTERVENTIONS = 24     # Number of complete control/intervention pairs (two generations each).
SWEEP_ALLOW_LARGE = False        # Set True only deliberately; prefer the Slurm runner for large grids.
SWEEP_SHOW_RAW_OUTPUTS = False

# ── 6. Completed durable layer-screen viewer (no model/GPU required) ──────
# Set this only after a batch runner has written RUN_COMPLETE and summary.json.
RUN_LAYER_SCREEN_RESULTS_VIEWER = False
LAYER_SCREEN_OUTPUT_DIR = os.environ.get('LAYERMCP_GPT_OSS_SCREEN_OUTPUT')

MODEL_CHOICES = {
    'tiny_cpu', 'loaded_model',
    'llama31_config_only', 'gpt_oss_config_only', 'qwen36_config_only',
    'gemma4_config_only', 'phi4_config_only',
    'llama31_checkpoint', 'gpt_oss_checkpoint', 'qwen36_checkpoint',
    'gemma4_checkpoint', 'phi4_checkpoint',
}
assert MODEL_CHOICE in MODEL_CHOICES


In [ ]:
from __future__ import annotations

import json
from dataclasses import fields
from importlib import import_module
from pathlib import Path

import torch
from torch import nn

from research.phase2.intervention import (
    AttentionInterventionError,
    attention_module_for_layer,
    available_attention_layers,
    available_attention_targets,
    temporary_attention_intervention,
)


In [ ]:
# Tiny, supported decoder layout for the CPU mechanics demonstration only.
class TinyAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(4, 4)
        self.k_proj = nn.Linear(4, 4)
        self.v_proj = nn.Linear(4, 4)
        self.o_proj = nn.Linear(4, 4)

    def forward(self, inputs):
        return self.o_proj(self.q_proj(inputs) + self.k_proj(inputs) + self.v_proj(inputs))


class TinyDecoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.self_attn = TinyAttention()
        self.mlp = nn.Linear(4, 4)


class TinySupportedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([TinyDecoderLayer(), TinyDecoderLayer()])


def print_table(title, rows):
    print('\n' + title)
    if not rows:
        print('  (none)')
        return
    columns = tuple(rows[0])
    widths = {key: max(len(str(key)), *(len(str(row.get(key, ''))) for row in rows)) for key in columns}
    print('  ' + ' | '.join(f'{key:<{widths[key]}}' for key in columns))
    print('  ' + '-+-'.join('-' * widths[key] for key in columns))
    for row in rows:
        values = [str(row.get(key, '')) for key in columns]
        print('  ' + ' | '.join(f'{value:<{widths[key]}}' for key, value in zip(columns, values)))


def target_matches(name, target):
    return target == 'attention_all' or name.startswith(target + '.')


def parameter_table(model, layer_index, target):
    module = attention_module_for_layer(model, layer_index)
    return [
        {
            'parameter': name, 'shape': tuple(parameter.shape),
            'dtype': str(parameter.dtype), 'device': str(parameter.device),
            'selected': target_matches(name, target),
        }
        for name, parameter in module.named_parameters(recurse=True)
    ]


def config_payload(checkpoint_path):
    config_path = checkpoint_path / 'config.json' if checkpoint_path is not None else None
    if config_path is not None and config_path.is_file():
        raw = json.loads(config_path.read_text(encoding='utf-8'))
        return raw.get('text_config', raw), 'local metadata: ' + str(config_path)
    return None, 'repository defaults (no checkpoint metadata read)'


def config_from_fields(config_class, checkpoint_path):
    raw, source = config_payload(checkpoint_path)
    if raw is None:
        return config_class(), source
    allowed = {item.name for item in fields(config_class)}
    return config_class(**{name: value for name, value in raw.items() if name in allowed}), source


def direct_rows(cfg, q_width, kv_width, o_width, include_v=True, label='q_proj'):
    rows = [
        {'parameter': label + '.weight', 'shape': (q_width, cfg.hidden_size), 'meaning': 'query projection'},
        {'parameter': 'k_proj.weight', 'shape': (kv_width, cfg.hidden_size), 'meaning': 'key projection'},
    ]
    if include_v:
        rows.append({'parameter': 'v_proj.weight', 'shape': (kv_width, cfg.hidden_size), 'meaning': 'value projection'})
    rows.append({'parameter': 'o_proj.weight', 'shape': (cfg.hidden_size, o_width), 'meaning': 'attention output projection'})
    return rows


def configuration_view(family, layer_index, checkpoint_path):
    # Architecture-derived shapes only; no model tensors are allocated.
    if family == 'llama31':
        from models.architectures.llama31_8b_pytorch.model import ModelConfigs
        cfg, source = config_from_fields(ModelConfigs, checkpoint_path)
        q, kv = cfg.num_attention_heads * cfg.head_dim, cfg.num_key_value_heads * cfg.head_dim
        module, kind = 'model.layers[i].self_attn', 'full attention'
        targets, rows = ('attention_all', 'q_proj', 'k_proj', 'v_proj', 'o_proj'), direct_rows(cfg, q, kv, q)
    elif family == 'gpt_oss':
        from models.architectures.gpt_oss_pytorch.model import ModelConfigs
        cfg, source = config_from_fields(ModelConfigs, checkpoint_path)
        q = cfg.num_attention_heads * cfg.head_dim
        qkv = cfg.head_dim * (cfg.num_attention_heads + 2 * cfg.num_key_value_heads)
        module, kind, targets = 'model.block[i].attn', 'full attention (fused QKV)', ('attention_all', 'qkv', 'out')
        rows = [
            {'parameter': 'qkv.weight', 'shape': (qkv, cfg.hidden_size), 'meaning': 'fused query/key/value projection'},
            {'parameter': 'qkv.bias', 'shape': (qkv,), 'meaning': 'fused QKV bias'},
            {'parameter': 'out.weight', 'shape': (cfg.hidden_size, q), 'meaning': 'attention output projection'},
            {'parameter': 'out.bias', 'shape': (cfg.hidden_size,), 'meaning': 'attention output bias'},
        ]
    elif family == 'gemma4':
        from models.architectures.gemma4_pytorch.model import ModelConfigs
        config_path = checkpoint_path / 'config.json' if checkpoint_path is not None else None
        if config_path is not None and config_path.is_file():
            cfg, source = ModelConfigs.from_json(str(config_path)), 'local metadata: ' + str(config_path)
        else:
            cfg, source = ModelConfigs(), 'repository defaults (no checkpoint metadata read)'
        global_layer = cfg.is_global(layer_index)
        head_dim, kv_heads = cfg.layer_head_dim(layer_index), cfg.layer_kv_heads(layer_index)
        q, kv = cfg.num_attention_heads * head_dim, kv_heads * head_dim
        module = 'model.layers[i].self_attn'
        kind = 'global attention; K and V share k_proj' if global_layer else 'sliding-window attention'
        has_v = not (global_layer and cfg.attention_k_eq_v)
        targets = ('attention_all', 'q_proj', 'k_proj', 'o_proj') if not has_v else ('attention_all', 'q_proj', 'k_proj', 'v_proj', 'o_proj')
        rows = direct_rows(cfg, q, kv, q, include_v=has_v) + [
            {'parameter': 'q_norm.weight', 'shape': (head_dim,), 'meaning': 'per-head query norm'},
            {'parameter': 'k_norm.weight', 'shape': (head_dim,), 'meaning': 'per-head key norm'},
        ]
    elif family == 'qwen36':
        from models.architectures.qwen36_pytorch.model import ModelConfigs
        cfg, source = config_from_fields(ModelConfigs, checkpoint_path)
        kind = cfg.layer_types[layer_index]
        if kind == 'full_attention':
            q, kv = cfg.num_attention_heads * cfg.head_dim * 2, cfg.num_key_value_heads * cfg.head_dim
            module, targets = 'model.layers[i].self_attn', ('attention_all', 'q_proj', 'k_proj', 'v_proj', 'o_proj')
            rows = direct_rows(cfg, q, kv, cfg.num_attention_heads * cfg.head_dim, label='q_proj (query + gate)')
        else:
            module, targets = 'model.layers[i].linear_attn', ('attention_all',)
            rows = [
                {'parameter': 'in_proj_qkv.weight', 'shape': ('architecture-dependent', cfg.hidden_size), 'meaning': 'linear-attention fused input'},
                {'parameter': 'conv1d.weight', 'shape': ('architecture-dependent',), 'meaning': 'linear-attention convolution'},
                {'parameter': 'out_proj.weight', 'shape': (cfg.hidden_size, 'architecture-dependent'), 'meaning': 'linear-attention output'},
            ]
    elif family == 'phi4':
        from models.architectures.phi4_pytorch.model import ModelConfigs
        config_path = checkpoint_path / 'config.json' if checkpoint_path is not None else None
        if config_path is not None and config_path.is_file():
            cfg, source = ModelConfigs.from_hf_config(str(config_path)), 'local metadata: ' + str(config_path)
        else:
            cfg, source = ModelConfigs(), 'repository defaults (no checkpoint metadata read)'
        q, kv = cfg.num_attention_heads * cfg.head_dim, cfg.num_key_value_heads * cfg.head_dim
        module, kind, targets = 'model.layers[i].self_attn', 'full attention (fused QKV)', ('attention_all', 'o_proj')
        rows = [
            {'parameter': 'qkv_proj.weight', 'shape': (q + 2 * kv, cfg.hidden_size), 'meaning': 'fused query/key/value projection'},
            {'parameter': 'o_proj.weight', 'shape': (cfg.hidden_size, q), 'meaning': 'attention output projection'},
        ]
    else:
        raise ValueError('Unknown model family: ' + family)
    layer_ids = tuple(range(cfg.num_hidden_layers))
    if layer_index not in layer_ids:
        raise ValueError('LAYER_INDEX=' + str(layer_index) + '; valid IDs are ' + str(layer_ids))
    return {'family': family, 'source': source, 'layer_ids': layer_ids, 'module': module, 'kind': kind, 'targets': targets, 'rows': rows}


In [ ]:
# Inspect a tiny/previously loaded model, or report an architecture view without weights.
checkpoint_path = Path(CHECKPOINT_DIR).expanduser() if CHECKPOINT_DIR is not None else None
active_model = None
active_generator = None
architecture = None

if MODEL_CHOICE == 'tiny_cpu':
    torch.manual_seed(0)
    active_model = TinySupportedModel().cpu()
elif MODEL_CHOICE == 'loaded_model':
    if LOADED_MODEL is None:
        raise ValueError('Set LOADED_MODEL to an already-loaded torch.nn.Module.')
    active_model = LOADED_MODEL
elif MODEL_CHOICE.endswith('_checkpoint'):
    if not LOAD_CHECKPOINT or checkpoint_path is None:
        raise ValueError('A checkpoint choice requires your CHECKPOINT_DIR and LOAD_CHECKPOINT = True.')
    if not checkpoint_path.is_dir():
        raise FileNotFoundError(checkpoint_path)
    if not torch.cuda.is_available():
        raise RuntimeError('Use an interactive GPU allocation before loading a large checkpoint.')
    family = MODEL_CHOICE.removesuffix('_checkpoint')
    packages = {
        'llama31': 'llama31_8b_pytorch', 'gpt_oss': 'gpt_oss_pytorch',
        'qwen36': 'qwen36_pytorch', 'gemma4': 'gemma4_pytorch', 'phi4': 'phi4_pytorch',
    }
    generator_class = import_module('models.architectures.' + packages[family] + '.inference').TokenGenerator
    active_generator = generator_class(checkpoint=str(checkpoint_path))
    active_model = active_generator.model
elif MODEL_CHOICE.endswith('_config_only'):
    architecture = configuration_view(MODEL_CHOICE.removesuffix('_config_only'), LAYER_INDEX, checkpoint_path)
else:
    raise AssertionError('Unhandled model choice: ' + MODEL_CHOICE)

if active_model is not None:
    layer_ids = available_attention_layers(active_model)
    if LAYER_INDEX not in layer_ids:
        raise ValueError('LAYER_INDEX=' + str(LAYER_INDEX) + '; valid IDs are ' + str(layer_ids))
    attention = attention_module_for_layer(active_model, LAYER_INDEX)
    available_targets = available_attention_targets(active_model, LAYER_INDEX)
    if ATTENTION_TARGET not in available_targets:
        raise ValueError('ATTENTION_TARGET=' + repr(ATTENTION_TARGET) + '; available: ' + str(available_targets))
    print('Actual loaded/tiny model')
    print('available layer IDs:', layer_ids)
    print('selected attention module:', type(attention).__qualname__)
    print('available targets:', available_targets)
    print_table('Selected attention parameters', parameter_table(active_model, LAYER_INDEX, ATTENTION_TARGET))
else:
    if ATTENTION_TARGET not in architecture['targets']:
        raise ValueError('ATTENTION_TARGET=' + repr(ATTENTION_TARGET) + '; expected available: ' + str(architecture['targets']))
    print('Configuration-only view; no model or weights were loaded.')
    print('family:', architecture['family'])
    print('configuration source:', architecture['source'])
    print('layer IDs:', architecture['layer_ids'])
    print('selected layer type:', architecture['kind'])
    print('attention module:', architecture['module'])
    print('available API targets:', architecture['targets'])
    print_table('Expected selected-layer attention parameters', architecture['rows'])


## Checkpoint quantization metadata versus loaded attention tensors

Checkpoint metadata describes how files were stored; it does **not** establish the dtype or writability of the attention tensors after this repository's loader has constructed the runtime model. The API rejects model-level 4-bit/8-bit flags and non-floating selected parameters. The zero-strength probe below invokes those structural checks without changing values.

A configuration-only view cannot answer this runtime question. Use an explicitly loaded local model when you need the answer for a real checkpoint.

In [ ]:
def quantization_metadata(path):
    if path is None:
        return {'status': 'no checkpoint directory supplied'}
    if not path.is_dir():
        return {'status': 'not a directory: ' + str(path)}
    findings = {}
    for filename in ('config.json', 'quantization_config.json', 'quantize_config.json'):
        candidate = path / filename
        if not candidate.is_file():
            continue
        try:
            payload = json.loads(candidate.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            findings[filename] = 'not valid JSON'
            continue
        quantized = {key: value for key, value in payload.items() if 'quant' in key.lower()}
        if quantized:
            findings[filename] = quantized
    return findings or {'status': 'no top-level quantization metadata found'}


print('checkpoint metadata (storage description, not runtime proof):')
print(quantization_metadata(checkpoint_path))

if active_model is None:
    print('No loaded model: actual selected-attention dtype and API acceptance are not established.')
else:
    selected_parameters = [(name, parameter) for name, parameter in attention.named_parameters(recurse=True) if target_matches(name, ATTENTION_TARGET)]
    flags = {name: bool(getattr(active_model, name, False)) for name in ('is_loaded_in_4bit', 'is_loaded_in_8bit')}
    print('model-level quantization flags:', flags)
    print('actual selected-attention types:', [(name, str(parameter.dtype), parameter.is_floating_point()) for name, parameter in selected_parameters])
    if RUN_ZERO_STRENGTH_ACCEPTANCE_PROBE:
        try:
            with temporary_attention_intervention(active_model, LAYER_INDEX, target=ATTENTION_TARGET, method='noise', strength=0.0, seed=0) as probe:
                print('full API result: accepted; zero-strength changed names:', probe.changed_parameter_names)
            print('exactly restored after probe:', probe.restored_exactly)
        except AttentionInterventionError as error:
            print('full API result: rejected:', error)


## Real selected-attention mechanics probe

Run this only after a local `<family>_checkpoint` has loaded. It checks the chosen settings-cell target twice: disabled mode must leave it unchanged, and enabled mode must alter it only while the context is active and restore it exactly afterward. This does not generate text or run a benchmark.

In [ ]:
if not RUN_REAL_INTERVENTION_PROBE:
    print('Real intervention probe disabled in the settings cell.')
elif active_model is None:
    print('Real intervention probe skipped: load a <family>_checkpoint model first.')
else:
    selected_before = {
        name: parameter.detach().clone()
        for name, parameter in attention.named_parameters(recurse=True)
        if target_matches(name, ATTENTION_TARGET)
    }
    with temporary_attention_intervention(
        active_model, LAYER_INDEX, target=ATTENTION_TARGET, method=INTERVENTION_METHOD,
        strength=INTERVENTION_STRENGTH, seed=INTERVENTION_SEED, enabled=False,
    ) as disabled_probe:
        disabled_changes = tuple(
            name for name, parameter in attention.named_parameters(recurse=True)
            if name in selected_before and not torch.equal(parameter, selected_before[name])
        )
    with temporary_attention_intervention(
        active_model, LAYER_INDEX, target=ATTENTION_TARGET, method=INTERVENTION_METHOD,
        strength=INTERVENTION_STRENGTH, seed=INTERVENTION_SEED, enabled=True,
    ) as active_probe:
        changed_inside = tuple(
            name for name, parameter in attention.named_parameters(recurse=True)
            if name in selected_before and not torch.equal(parameter, selected_before[name])
        )
    restored_exactly = all(
        torch.equal(parameter, selected_before[name])
        for name, parameter in attention.named_parameters(recurse=True) if name in selected_before
    )
    print({
        'disabled_changed_names': disabled_changes,
        'method': INTERVENTION_METHOD,
        'strength': INTERVENTION_STRENGTH,
        'seed': INTERVENTION_SEED,
        'changed_inside_context': changed_inside,
        'restored_exactly_after_context': restored_exactly and active_probe.restored_exactly,
    })

## Tiny CPU mechanics: disabled, noise, replacement, and exact restoration

This deliberately targets only `q_proj` in layer 1. Its MLP, K/V/O projections, and every other layer must remain bit-for-bit unchanged. The context manager restores the original values after each mode.

In [ ]:
def clone_parameters(model):
    return {name: parameter.detach().clone() for name, parameter in model.named_parameters()}


def exact_match(model, saved):
    return all(torch.equal(parameter, saved[name]) for name, parameter in model.named_parameters())


torch.manual_seed(123)
demo_model = TinySupportedModel().cpu()
demo_layer, demo_target = 1, 'q_proj'
before = clone_parameters(demo_model)
inputs = torch.ones(2, 4)
baseline = demo_model.layers[demo_layer].self_attn(inputs).detach().clone()

with temporary_attention_intervention(demo_model, demo_layer, target=demo_target, method='noise', strength=0.5, seed=7, enabled=False) as disabled:
    disabled_output = demo_model.layers[demo_layer].self_attn(inputs).detach().clone()
assert torch.equal(baseline, disabled_output)
assert disabled.changed_parameter_names == ()
assert exact_match(demo_model, before)

results = {'disabled': 'no-op; identical output and exact values retained'}
for method in ('noise', 'replace'):
    with temporary_attention_intervention(demo_model, demo_layer, target=demo_target, method=method, strength=0.5, seed=7, enabled=True) as intervention:
        active = clone_parameters(demo_model)
        assert intervention.changed_parameter_names
        for name, value in active.items():
            targeted = name.startswith('layers.' + str(demo_layer) + '.self_attn.' + demo_target + '.')
            assert torch.equal(value, before[name]) is (not targeted), name
        results[method] = {'changed_attention_parameters': intervention.changed_parameter_names}
    assert intervention.restored_exactly
    assert exact_match(demo_model, before)
    results[method]['restored_after_context'] = intervention.restored_exactly

print(results)


## One saved GPT-OSS sample: control versus intervention

Set `RUN_PAIRED_SAMPLE_COMPARISON = True` only after loading `gpt_oss_checkpoint` on a GPU and setting `LAYERMCP_GPT_OSS_SOURCE_RUN` in your ignored `.env`. The cell runs **one** saved sample twice through the existing GPT-OSS Harmony parser and baseline evaluator: unchanged control, then the selected temporary intervention. It executes the predicted tool in the evaluator's normal isolated session, shows both raw outputs and scores, and requires exact restoration. It does not write a model checkpoint or start a sweep.

Start with `math_v1_calculator_easy_001` to check that the unchanged control reproduces a known baseline success and that the intervention preserves or harms it. Use a known baseline failure such as `finance_upstream_inspired_finance_parse_xbrl_004` only after that, to test whether a setting can change a failure. This is an exploratory one-sample comparison, not evidence that a setting generalizes.

In [ ]:
if not RUN_PAIRED_SAMPLE_COMPARISON:
    print('One-sample paired comparison disabled in the settings cell.')
elif MODEL_CHOICE != 'gpt_oss_checkpoint' or active_generator is None:
    print('One-sample comparison requires MODEL_CHOICE = gpt_oss_checkpoint and LOAD_CHECKPOINT = True.')
elif not PAIRED_SOURCE_RUN_DIR:
    print('Set LAYERMCP_GPT_OSS_SOURCE_RUN in the ignored .env file before running this cell.')
else:
    from research.phase2.gpt_oss_intervention_eval import evaluate_one_saved_example

    paired_record = evaluate_one_saved_example(
        generator=active_generator,
        source_run_dir=Path(PAIRED_SOURCE_RUN_DIR),
        sample_id=PAIRED_SAMPLE_ID,
        layer_index=LAYER_INDEX,
        seed=INTERVENTION_SEED,
        method=INTERVENTION_METHOD,
        target=ATTENTION_TARGET,
        strength=INTERVENTION_STRENGTH,
    )
    print('sample:', paired_record['sample_id'])
    print('expected tool:', paired_record['control']['expected_tool'])
    print('expected arguments:', json.dumps(paired_record['control']['expected_args'], sort_keys=True))

    comparison_rows = []
    for condition_name in ('control', 'intervention'):
        condition = paired_record[condition_name]
        verification = paired_record[condition_name + '_verification']
        comparison_rows.append({
            'condition': condition_name,
            'selected_tool': condition['selected_tool'],
            'tool_correct': condition['tool_selection_correct'],
            'args_exact': condition['argument_match_correct'],
            'called_tool': condition['called_tool'],
            'execution_success': condition['execution_success'],
            'final_outcome': condition['final_outcome_correct'],
            'invalid_output': condition['invalid_output'],
            'no_call': condition['no_call_outcome'],
            'changed_parameters': verification['changed_attention_parameter_names'],
            'restored_exactly': verification['restored_exactly'] if verification['enabled'] else 'not applicable (disabled)',
        })
    print_table('Paired evaluator comparison', comparison_rows)

    for condition_name in ('control', 'intervention'):
        condition = paired_record[condition_name]
        print('\n--- ' + condition_name.upper() + ' RAW MODEL OUTPUT ---')
        print(condition['raw_model_output'])
        print('selected arguments:', json.dumps(condition['selected_args'], sort_keys=True))
        print('tool result:', condition['tool_result'])


## Small paired GPT-OSS sweep

Set `RUN_PAIRED_SWEEP = True` only after a local `gpt_oss_checkpoint` has loaded on a GPU. Every planned setting runs the same saved sample twice: an unchanged control and a temporary intervention. The table therefore shows whether a configuration preserved, harmed, or changed the selected examples.

`SWEEP_LAYERS = 'all'` expands to every safely identifiable layer; `SWEEP_TARGETS = 'all'` expands only to targets actually safe at each layer. The default cap is 24 complete pairs (48 generations). A larger request stops before generation unless `SWEEP_ALLOW_LARGE = True`; use the Slurm runner for an exhaustive model-wide grid because an interactive allocation has a hard time limit.

In [ ]:
if not RUN_PAIRED_SWEEP:
    print('Small paired sweep disabled in the settings cell.')
elif MODEL_CHOICE != 'gpt_oss_checkpoint' or active_generator is None:
    print('Small paired sweep requires MODEL_CHOICE = gpt_oss_checkpoint and LOAD_CHECKPOINT = True.')
elif not PAIRED_SOURCE_RUN_DIR:
    print('Set LAYERMCP_GPT_OSS_SOURCE_RUN in the ignored .env file before running this cell.')
else:
    from research.phase2.gpt_oss_intervention_eval import evaluate_one_saved_example
    from research.phase2.paired_sweep import PairedSweepPlanError, plan_paired_sweep

    safe_layers = available_attention_layers(active_model)
    safe_targets = {layer_index: available_attention_targets(active_model, layer_index) for layer_index in safe_layers}
    try:
        sweep_settings = plan_paired_sweep(
            sample_ids=SWEEP_SAMPLE_IDS, layers=SWEEP_LAYERS, targets=SWEEP_TARGETS,
            methods=SWEEP_METHODS, strengths=SWEEP_STRENGTHS, seeds=SWEEP_SEEDS,
            available_layers=safe_layers, available_targets_by_layer=safe_targets,
            max_interventions=SWEEP_MAX_INTERVENTIONS, allow_large=SWEEP_ALLOW_LARGE,
        )
    except PairedSweepPlanError as error:
        print('Sweep plan rejected:', error)
    else:
        print('Planned control/intervention pairs:', len(sweep_settings))
        print('Each pair generates once unchanged and once intervened.')
        sweep_rows = []
        sweep_records = []
        for number, setting in enumerate(sweep_settings, start=1):
            print('Running pair ' + str(number) + '/' + str(len(sweep_settings)) + ':', setting)
            paired_record = evaluate_one_saved_example(
                generator=active_generator, source_run_dir=Path(PAIRED_SOURCE_RUN_DIR),
                sample_id=setting.sample_id, layer_index=setting.layer_index,
                seed=setting.seed, method=setting.method, target=setting.target,
                strength=setting.strength,
            )
            sweep_records.append(paired_record)
            control, intervention = paired_record['control'], paired_record['intervention']
            verification = paired_record['intervention_verification']
            if not verification['restored_exactly']:
                raise RuntimeError('Intervention did not restore exactly: ' + repr(verification))
            sweep_rows.append({
                'sample_id': setting.sample_id, 'layer': setting.layer_index, 'target': setting.target,
                'method': setting.method, 'strength': setting.strength, 'seed': setting.seed,
                'expected_tool': control['expected_tool'],
                'selected_tools': str(control['selected_tool']) + ' -> ' + str(intervention['selected_tool']),
                'tool_correct': str(control['tool_selection_correct']) + ' -> ' + str(intervention['tool_selection_correct']),
                'args_exact': str(control['argument_match_correct']) + ' -> ' + str(intervention['argument_match_correct']),
                'execution': str(control['execution_success']) + ' -> ' + str(intervention['execution_success']),
                'final_outcome': str(control['final_outcome_correct']) + ' -> ' + str(intervention['final_outcome_correct']),
                'invalid_output': str(control['invalid_output']) + ' -> ' + str(intervention['invalid_output']),
                'no_call': str(control['no_call_outcome']) + ' -> ' + str(intervention['no_call_outcome']),
                'raw_output_changed': control['raw_model_output'] != intervention['raw_model_output'],
                'restored_exactly': verification['restored_exactly'],
            })
            if SWEEP_SHOW_RAW_OUTPUTS:
                print('--- CONTROL RAW OUTPUT ---\n' + control['raw_model_output'])
                print('--- INTERVENTION RAW OUTPUT ---\n' + intervention['raw_model_output'])
        print_table('Small paired sweep comparison', sweep_rows)
        print({
            'control_final_outcomes': sum(record['control']['final_outcome_correct'] for record in sweep_records),
            'intervention_final_outcomes': sum(record['intervention']['final_outcome_correct'] for record in sweep_records),
            'raw_outputs_changed': sum(row['raw_output_changed'] for row in sweep_rows),
            'pairs_restored_exactly': sum(row['restored_exactly'] for row in sweep_rows),
        })


## Completed GPT-OSS layer-screen results viewer

A full layer screen belongs in the durable runner, not this interactive kernel. After a completed batch run, set `LAYERMCP_GPT_OSS_SCREEN_OUTPUT` in your ignored `.env`, set `RUN_LAYER_SCREEN_RESULTS_VIEWER = True`, and run this cell. It reads only `RUN_COMPLETE`, `screen_plan.json`, and `summary.json`; it does not load a model, allocate a GPU, generate text, or change weights.

The table reports each layer's control and intervention seed means plus sample standard deviations. A null standard deviation means there was only one defined seed value, not that the result was perfectly stable.

In [ ]:
if not RUN_LAYER_SCREEN_RESULTS_VIEWER:
    print('Completed layer-screen results viewer disabled in the settings cell.')
elif not LAYER_SCREEN_OUTPUT_DIR:
    print('Set LAYERMCP_GPT_OSS_SCREEN_OUTPUT in ignored .env to a completed runner output directory.')
else:
    screen_output = Path(LAYER_SCREEN_OUTPUT_DIR).expanduser()
    if not (screen_output / 'RUN_COMPLETE').is_file():
        raise RuntimeError('Refusing to summarize an incomplete screen: RUN_COMPLETE is missing.')
    plan = json.loads((screen_output / 'screen_plan.json').read_text(encoding='utf-8'))
    summary = json.loads((screen_output / 'summary.json').read_text(encoding='utf-8'))
    print('Completed screen:', screen_output)
    print({
        'layers': plan['selected_layers'],
        'samples': plan['sample_ids'],
        'seeds': plan['seeds'],
        'target': plan['target'],
        'method': plan['method'],
        'strength': plan['strength'],
        'complete_pairs': plan['total_pair_count'],
        'total_generations': plan['total_generation_count'],
    })

    metric_names = ('final_outcome_accuracy', 'tool_choice_accuracy', 'exact_argument_match_accuracy', 'execution_success_rate', 'invalid_output_rate', 'no_call_outcome_rate')
    screen_rows = []
    for layer, layer_summary in sorted(summary['by_layer'].items(), key=lambda item: int(item[0])):
        row = {
            'layer': int(layer),
            'seed_count': layer_summary['seed_count'],
            'restored_exactly': layer_summary['intervention_integrity']['all_restored_exactly'],
        }
        for metric in metric_names:
            control = layer_summary['conditions']['control'][metric]
            intervention = layer_summary['conditions']['intervention'][metric]
            row['control ' + metric] = control['mean']
            row['intervention ' + metric] = intervention['mean']
            row['delta ' + metric] = (
                None if control['mean'] is None or intervention['mean'] is None
                else intervention['mean'] - control['mean']
            )
            row['intervention ' + metric + ' sample_std'] = intervention['sample_standard_deviation']
        screen_rows.append(row)
    print_table('Completed GPT-OSS layer-screen summary', screen_rows)


## Real inference call site

After inspecting an **actually loaded** model, put the existing repository inference call inside the context manager. Use the same prompt, native tokenizer/Harmony format, and generation settings for an unchanged control (`enabled=False`) and an intervention. The one-sample GPT-OSS cell above calls the paired evaluator for you; this generic sketch explains the lower-level call site.

```python
with temporary_attention_intervention(
    model, LAYER_INDEX, target=ATTENTION_TARGET, method='noise',
    strength=0.01, seed=1234, enabled=True,
):
    result = existing_generator.generate(existing_prompt)
# original selected parameters are restored here, even if generation raises
```

GPT-OSS exposes fused `qkv` and `out`, not separate Q/K/V. Phi exposes only individual `o_proj`; its `qkv_proj` remains fused. A Qwen linear-attention layer intentionally supports `attention_all` only today.